# HW10 T1-T4: differential evolution and fitting
**Algorithmics 2026 | version 2026-11-02 | helpers, not completed solutions**

T1: a line and its landscape. T2: curves and outliers. T3: crossover and controlled experiments. T4: unknown frequencies.

Run cells from top to bottom. The demonstrations run immediately; assessed
experiment cells are gated by `RUN_T... = False` until your solver is ready.
No downloads, Drive mount or additional helper files are needed. Colab
normally includes NumPy, Matplotlib and Pillow; in local Jupyter install
these helper dependencies if missing.

Your optimiser uses standard Python (`random`, `math`, lists). Do not call
optimisation, evolutionary or regression-fitting packages in T1-T5.
Helper dependencies handle data, scoring and visualisation only. Packages
become allowed in bonus B2. Write your own report; a notebook link is not
a report. Known truth and held-out data must never select a search candidate.


In [ ]:
%matplotlib inline
from pathlib import Path
import random
import math
import numpy as np
import matplotlib.pyplot as plt

OUTPUT = Path('HW10_fitting_outputs')
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
print("Output folder:", OUTPUT.resolve())
print("NumPy:", np.__version__)


### What does the evaluation budget mean?

**Budget = maximum number of fully scored candidate solutions per run.**
It is not a time limit, generation count, target error or grading points.
A fresh seed/objective/configuration gets a fresh budget.

- One parameter vector scored on ALL training points = **one evaluation**.
- One complete 50-circle image rendered and scored = **one evaluation**.
- Initial candidates and rejected trials count. Recalculating a score
  counts again; reusing a cached score does not. N candidates in a batch
  count as N evaluations, even when computed in one library call.

Example: population 40 and budget 8,000 leave **7,960 evaluations after
initialisation**. With 40 new trials per generation this is 199 further
generations: `40 + 199*40 = 8000`.

Inside your solver, **always obtain search fitness with
`score = run(candidate)`**. This scores and increments `run.evaluations`.
Check `run.remaining` before another evaluation. The helper raises
`BudgetExhausted` if you exceed the limit; raw scoring calls would bypass
it and must not be used for search. The provided runners create the counter
automatically. The saved CSV/JSON logs record actual usage and progress.

**Choose your own evaluation budgets.** Set them before a comparison,
briefly justify them, and keep the same allowance across compared
methods/settings and seeds on the same problem. Different tasks may use
different budgets. More evaluations do not by themselves earn more credit.
The suggested starting values are T1-T3: 8,000; T4: 16,000; T5: 3,000,
not requirements. Use your chosen budget for the main comparisons,
including a partial final generation. T3's target error 0.10 is a milestone,
not an early-stop condition or grading threshold.
Report chosen/used evaluations, your reason, final best fitness and search seconds;
explain any failed or shorter run. Diagnostic landscapes, post-search
held-out checks and final export verification are outside the search
budget and must not select candidates. Plot/display time is separate.


### Experiment budgets: change them here

These are suggested starting values, not required budgets. Choose smaller
or larger values for your experiments and briefly explain your choice.
Keep equal budgets within each comparison and report the value used.
Re-run the relevant experiment cells after changing a setting. All student
experiment calls below use these variables. Budget must allow you to
observe search progress, not just initialise the population.

For a quick test, try 400 evaluations. The budget must at least cover the
initial population (up to 80 in T3). Read quality at 1,000/2,000/4,000/8,000
from one saved long-run history rather than rerunning just for each cutoff.


In [ ]:
FIT_BUDGET = 8_000    # Suggested T1-T3 budget; choose your own, including initial scoring
HARD_BUDGET = 16_000  # Suggested T4 budget; choose your own


## T1. Fit a line: mean versus median absolute error

A candidate is `[intercept, slope]`. Fit `y = b + a*x` using two separate
runs: MAE and MedAE. Lower is better. Start with population 40, F=0.8,
CR=1 (the complete donor), seed 0, and your chosen `FIT_BUDGET` per objective.

The following bookkeeping counts every candidate and stores a COPY of
the best. It supplies no search operators. Call `run(candidate)` for all
initial candidates and trials; retain cached scores for survivors.
`run.remaining` prevents exceeding the budget. `run_fitting` starts the
timer before population initialisation and calls `finish()` afterwards.


In [ ]:
#@title Helper definitions (run this cell)
"""HW10 bookkeeping only: no evolutionary search operators or solvers."""

import csv
from contextlib import contextmanager
import json
import math
from pathlib import Path
import random
from time import perf_counter


class BudgetExhausted(RuntimeError):
    """The next candidate would exceed the agreed evaluation budget."""


def checked_bounds(bounds):
    result = tuple((float(low), float(high)) for low, high in bounds)
    if not result or any(not math.isfinite(low) or not math.isfinite(high)
                         or low >= high for low, high in result):
        raise ValueError("Each bound must be finite and low < high.")
    return result


def checked_candidate(candidate, bounds):
    values = tuple(float(value) for value in candidate)
    if len(values) != len(bounds):
        raise ValueError(f"Expected {len(bounds)} parameters, got {len(values)}.")
    if any(not math.isfinite(value) or not low <= value <= high
           for value, (low, high) in zip(values, bounds)):
        raise ValueError("Candidate is nonfinite or outside the agreed bounds.")
    return values


def initial_population(bounds, size=40, seed=0):
    """Independent initialisation RNG: settings cannot change this population."""
    bounds = checked_bounds(bounds)
    if not isinstance(size, int) or size < 1:
        raise ValueError("Population size must be a positive integer.")
    rng = random.Random(seed)
    return [[rng.uniform(low, high) for low, high in bounds] for _ in range(size)]


class FitnessRun:
    """Count candidate evaluations, keep a copied best, and log search time.

    Call start() immediately before population initialisation, then call this
    object for EVERY evaluated candidate. Call finish() after the search loop.
    A pause() context excludes display/export but refuses fitness evaluations.
    No selection, mutation, crossover or optimisation is implemented here.
    """

    def __init__(self, objective, bounds, budget, *, label, objective_name,
                 seed=0, settings=None, snapshot_every=200):
        if not isinstance(budget, int) or budget < 1:
            raise ValueError("Budget must be a positive integer.")
        if not isinstance(snapshot_every, int) or snapshot_every < 1:
            raise ValueError("Snapshot interval must be a positive integer.")
        self.objective = objective
        self.bounds = checked_bounds(bounds)
        self.budget, self.label = budget, str(label)
        self.objective_name, self.seed = str(objective_name), int(seed)
        self.settings = dict(settings or {})
        self.snapshot_every = snapshot_every
        self.evaluations = 0
        self.best = None
        self.best_fitness = math.inf
        self.history, self.snapshots = [], []
        self._start = None
        self._end = None
        self._paused = 0.0
        self._pause_start = None

    def start(self):
        if self._start is None:
            self._start = perf_counter()
        return self

    @property
    def remaining(self):
        return self.budget - self.evaluations

    @property
    def seconds(self):
        if self._start is None:
            return 0.0
        end = self._end if self._end is not None else perf_counter()
        pending = end - self._pause_start if self._pause_start is not None else 0.0
        return max(0.0, end - self._start - self._paused - pending)

    @contextmanager
    def pause(self):
        if self._pause_start is not None:
            raise RuntimeError("Do not nest pause contexts.")
        active = self._start is not None and self._end is None
        if active:
            self._pause_start = perf_counter()
        try:
            yield
        finally:
            if active:
                self._paused += perf_counter() - self._pause_start
                self._pause_start = None

    def __call__(self, candidate):
        if self.remaining <= 0:
            raise BudgetExhausted(f"Budget of {self.budget} candidates exhausted.")
        if self._end is not None or self._pause_start is not None:
            raise RuntimeError("Cannot score while paused or after finish().")
        values = checked_candidate(candidate, self.bounds)
        self.start()
        score = float(self.objective(values))
        if not math.isfinite(score):
            raise ValueError("Fitness must be finite.")
        self.evaluations += 1
        if score < self.best_fitness:
            self.best_fitness, self.best = score, values
        self.history.append({"evaluation": self.evaluations,
                             "seconds": self.seconds, "fitness": score,
                             "best_fitness": self.best_fitness})
        if self.evaluations == 1 or self.evaluations % self.snapshot_every == 0:
            self.snapshot()
        return score

    def snapshot(self):
        if self.best is not None:
            item = {"evaluation": self.evaluations, "best": list(self.best),
                    "fitness": self.best_fitness}
            if self.snapshots and self.snapshots[-1]["evaluation"] == self.evaluations:
                self.snapshots[-1] = item
            else:
                self.snapshots.append(item)

    def finish(self):
        if self._pause_start is not None:
            raise RuntimeError("Finish outside a pause context.")
        if self._end is None:
            self.start()
            self._end = perf_counter()
            self.snapshot()
        return self.best

    def summary(self, target=None):
        hit = next((row for row in self.history if target is not None
                    and row["best_fitness"] <= target), None)
        return {"label": self.label, "seed": self.seed,
                "objective": self.objective_name, "settings": self.settings,
                "budget": self.budget, "evaluations": self.evaluations,
                "search_seconds": self.seconds,
                "best_fitness": None if self.best is None else self.best_fitness,
                "best": None if self.best is None else list(self.best),
                "target": target, "reached": hit is not None,
                "target_evaluation": None if hit is None else hit["evaluation"],
                "target_seconds": None if hit is None else hit["seconds"]}

    def save(self, stem, target=None):
        """Save every evaluation plus metadata; finish before exporting."""
        if self._end is None:
            raise RuntimeError("Call finish() before saving the run.")
        stem = Path(stem)
        stem.parent.mkdir(parents=True, exist_ok=True)
        with stem.with_suffix(".csv").open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=[
                "evaluation", "seconds", "fitness", "best_fitness"])
            writer.writeheader()
            writer.writerows(self.history)
        data = self.summary(target)
        data.update(bounds=self.bounds, snapshots=self.snapshots)
        stem.with_suffix(".json").write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")


def plot_progress(runs, title="Search progress"):
    import matplotlib.pyplot as plt

    runs = list(runs)
    if not runs or any(not run.history for run in runs):
        raise ValueError("Supply at least one evaluated run.")
    if len({run.objective_name for run in runs}) != 1:
        raise ValueError("Plot different objectives separately: their scores differ.")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), layout="constrained")
    for run in runs:
        for ax, key, label in zip(axes, ("evaluation", "seconds"),
                                  ("Fitness evaluations", "Search time (seconds)")):
            ax.plot([row[key] for row in run.history],
                    [row["best_fitness"] for row in run.history], label=run.label)
            ax.set(xlabel=label, ylabel=f"Best {run.objective_name} (lower is better)")
            ax.grid(alpha=0.2)
    axes[1].legend(fontsize=8)
    fig.suptitle(title)
    return fig


def experiment_table(runs, target=None):
    """All trials, including failures; no winner selection or solver calls."""
    rows = [run.summary(target) for run in runs]
    for row in rows:
        score = row["best_fitness"]
        score_text = "none" if score is None else f"{score:.6f}"
        hit = row["target_evaluation"] if row["reached"] else "not reached"
        target_text = "" if target is None else f"  target={hit}"
        print(f"{row['label']:<24} seed={row['seed']}  fitness={score_text}  "
              f"evals={row['evaluations']}  seconds={row['search_seconds']:.3f}{target_text}")
    return rows


### T1: data, objectives and a runner for your solver


In [ ]:
#@title Helper definitions (run this cell)
import hashlib
import json
import numpy as np
import matplotlib.pyplot as plt


POLYNOMIALS = {
    "line": [0.5, 1.5],
    "cubic": [0.1, -2.4, 0.2, 2.7],
    "quintic": [0.1, 1.8, -0.8, -5.0, 0.4, 3.5],
}


def model_values(parameters, x, model="polynomial"):
    values = np.asarray(parameters, dtype=float)
    x = np.asarray(x, dtype=float)
    if values.ndim != 1 or not np.all(np.isfinite(values)):
        raise ValueError("Parameters must be a finite vector.")
    if model == "polynomial":
        if not 2 <= len(values) <= 6:
            raise ValueError("Use 2-6 polynomial coefficients, lowest power first.")
        return np.polynomial.polynomial.polyval(x, values)
    waves = {"one-wave": 1, "two-wave": 2}.get(model)
    if waves is None or len(values) != 3 + 3 * waves:
        raise ValueError("Wrong model or parameter count.")
    result = np.polynomial.polynomial.polyval(x, values[:3])
    for start in range(3, len(values), 3):
        amplitude, frequency, phase = values[start:start + 3]
        result = result + amplitude * np.sin(frequency * x + phase)
    return result


def observations(model, truth, bounds, *, name, seed=2026, n=100, noise=0.1,
                 outlier_fraction=0.0, sampling="uniform", noise_pattern="constant",
                 outlier_direction="up"):
    if not isinstance(n, int) or n < 10:
        raise ValueError("Use at least ten observations.")
    if not np.isfinite(noise) or noise < 0 or not 0 <= outlier_fraction <= 0.4:
        raise ValueError("Noise >= 0; outlier fraction in [0,0.4].")
    if sampling not in ("uniform", "gap", "clustered"):
        raise ValueError("Sampling: uniform, gap or clustered.")
    if noise_pattern not in ("constant", "increasing") or outlier_direction not in ("up", "mixed"):
        raise ValueError("Unknown noise pattern or outlier direction.")
    rx, rn, ro = [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(3)]
    if sampling == "uniform":
        x = rx.uniform(-1, 1, n)
    elif sampling == "gap":
        x = rx.choice([-1, 1], n) * rx.uniform(0.4, 1, n)
    else:
        x = np.clip(rx.normal(rx.choice([-0.65, 0.65], n, p=[0.75, 0.25]), 0.12), -1, 1)
    sigma = noise if noise_pattern == "constant" else noise * (0.2 + 1.9 * (x + 1))
    clean = model_values(truth, x, model)
    y = clean + rn.normal(size=n) * sigma
    order = ro.permutation(n)
    shifts = ro.uniform(3, 6, n)
    if outlier_direction == "mixed":
        shifts *= ro.choice([-1, 1], n)
    mask = np.zeros(n, dtype=bool)
    mask[order[:round(n * outlier_fraction)]] = True
    y[mask] += shifts[mask]
    x_valid = np.linspace(-1, 1, 501)
    return {"name": name, "model": model, "x": x, "y": y,
            "truth": list(truth), "bounds": list(bounds), "seed": seed,
            "outlier_fraction": outlier_fraction, "outlier_mask": mask,
            "x_valid": x_valid, "y_valid": model_values(truth, x_valid, model)}


def polynomial_data(case="line", **options):
    if case not in POLYNOMIALS:
        raise ValueError("Choose line, cubic or quintic.")
    truth = POLYNOMIALS[case]
    limit = 3 if case == "line" else 6
    return observations("polynomial", truth, [(-limit, limit)] * len(truth),
                        name=case, **options)


def error_value(y, predictions, objective="MAE"):
    residuals = np.abs(np.asarray(y) - np.asarray(predictions))
    if objective == "MAE":
        return float(np.mean(residuals))
    if objective == "MedAE":
        return float(np.median(residuals))
    raise ValueError("Choose MAE or MedAE.")


def training_objective(data, objective="MAE"):
    # Keep only training values in the scoring closure, not the generating truth.
    x, y, model = data["x"].copy(), data["y"].copy(), data["model"]
    if objective not in ("MAE", "MedAE"):
        raise ValueError("Choose MAE or MedAE.")
    return lambda parameters: error_value(y, model_values(parameters, x, model), objective)


def fit_metrics(data, parameters):
    result = {}
    for split, xkey, ykey in (("training", "x", "y"), ("held_out", "x_valid", "y_valid")):
        predictions = model_values(parameters, data[xkey], data["model"])
        for objective in ("MAE", "MedAE"):
            result[f"{split}_{objective}"] = error_value(data[ykey], predictions, objective)
    return result


def fitting_data_digest(data):
    """Identify the training problem so cached results cannot cross datasets."""
    payload = {"model": data["model"]}
    for key in ("x", "y", "bounds"):
        payload[key] = np.asarray(data[key], dtype=float).tolist()
    encoded = json.dumps(payload, sort_keys=True, allow_nan=False).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def run_fitting(solver, data, objective="MAE", *, seed=0, population=40,
                F=0.8, CR=1.0, budget=8000, label=None):
    """Call YOUR solver, which must score its initial population via run()."""
    if not isinstance(population, int) or population < 4:
        raise ValueError("DE needs at least four population members.")
    run = FitnessRun(training_objective(data, objective), data["bounds"], budget,
                     label=label or f"{objective}, seed {seed}", objective_name=objective,
                     seed=seed, settings={"population": population, "F": F, "CR": CR,
                                          "data": data["name"], "data_seed": data["seed"],
                                          "outliers": data["outlier_fraction"],
                                          "training_sha256": fitting_data_digest(data)})
    if run.budget < population:
        raise ValueError("Budget must cover the entire initial population.")
    run.start()
    initial = initial_population(data["bounds"], population, seed)
    solver(run, initial, F=F, CR=CR, seed=10000 + seed)
    run.finish()
    if run.evaluations == 0:
        raise RuntimeError("Your solver did not evaluate any candidates.")
    return run


### T1: curve and landscape plots

`plot_landscapes` computes a diagnostic grid in intercept/slope space.
This is NOT an optimiser and its grid scores are not counted as search.
Inspect landscapes after fitting; do not use them to initialise or choose
candidates. For your final report pass `{'MAE': run_mean.best,
'MedAE': run_median.best}` to mark the returned solutions.


In [ ]:
def plot_fit(data, fits=None, title=None, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
    mask = data["outlier_mask"]
    ax.scatter(data["x"][~mask], data["y"][~mask], s=20, color="#54828A",
               alpha=0.7, label="Observed points")
    if np.any(mask):
        ax.scatter(data["x"][mask], data["y"][mask], s=30, marker="x",
                   color="#B95024", label="Injected outliers (diagnostic only)")
    ax.plot(data["x_valid"], data["y_valid"], color="#222222", lw=2,
            label="Original curve (evaluation only)")
    for index, (label, parameters) in enumerate((fits or {}).items()):
        ax.plot(data["x_valid"], model_values(parameters, data["x_valid"], data["model"]),
                linestyle=["--", "-.", ":"][index % 3], label=label, lw=1.8)
    ax.set(xlabel="x", ylabel="y", title=title or
           f"{data['name']}: {100 * data['outlier_fraction']:.0f}% outliers")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.15)
    return ax


def plot_landscapes(data, solutions=None, resolution=51, contours=False):
    """Diagnostic grid ONLY: no optimiser calls, counter updates or best-grid output."""
    if data["model"] != "polynomial" or len(data["bounds"]) != 2:
        raise ValueError("The line landscape has exactly two parameters [b,a].")
    if not isinstance(resolution, int) or not 10 <= resolution <= 101:
        raise ValueError("Choose a grid resolution from 10 to 101.")
    b, a = np.meshgrid(*(np.linspace(low, high, resolution) for low, high in data["bounds"]))
    residuals = np.abs(data["y"] - b[..., None] - a[..., None] * data["x"])
    fig = plt.figure(figsize=(12, 5.3), layout="constrained")
    for index, objective in enumerate(("MAE", "MedAE"), 1):
        height = (np.mean if objective == "MAE" else np.median)(residuals, axis=-1)
        ax = fig.add_subplot(1, 2, index, **({} if contours else {"projection": "3d"}))
        if contours:
            artist = ax.contourf(b, a, height, levels=20, cmap="viridis")
            fig.colorbar(artist, ax=ax, label=objective, shrink=0.8)
        else:
            ax.plot_surface(b, a, height, cmap="viridis", alpha=0.9, linewidth=0)
            ax.set_zlabel(f"{objective} (lower is better)", labelpad=8)
            ax.view_init(elev=28, azim=-55)
        point = (solutions or {}).get(objective)
        if point is not None:
            args = [point[0], point[1]]
            if not contours:
                args.append(training_objective(data, objective)(point))
            ax.scatter(*args, color="#E34234", edgecolor="white", s=70, zorder=5)
        ax.set(xlabel="Intercept b", ylabel="Slope a", title=objective)
    fig.suptitle("T1: training fitness landscape (diagnostic, not search)")
    return fig


In [ ]:
line_data = polynomial_data("line")
plot_fit(line_data, {"Hand-picked illustration, not a fit": [0.0, 1.0]})
plt.savefig(OUTPUT / "T1_line_demo.png", dpi=140)
plt.show()
landscape = plot_landscapes(line_data)
landscape.savefig(OUTPUT / "T1_landscapes.png", dpi=140)
plt.show()


### T1: check the fitness logger before writing DE

The next cell scores just three HAND-PICKED candidates to illustrate
the counter and both progress axes. It is not a search result, nor a
reference solution. Times in this tiny smoke test are not a benchmark.


In [ ]:
logger_demo = FitnessRun(training_objective(line_data, "MAE"), line_data["bounds"],
                         3, label="Logger demonstration only", objective_name="MAE")
logger_demo.start()
for candidate in ([0.0, 0.0], [0.0, 1.0], [0.2, 1.2]):
    score = logger_demo(candidate)
    print(f"Fitness {score:.5f}; used {logger_demo.evaluations}/{logger_demo.budget}; "
          f"remaining {logger_demo.remaining}")
logger_demo.finish()
plot_progress([logger_demo], "T1: three hand-picked candidates, not DE")
plt.savefig(OUTPUT / "T1_logger_demo.png", dpi=140)
plt.show()
print(logger_demo.summary())


### T1: your implementation

Fill the function below. It receives an independent initial population
of Python lists and a callable `run` with `.bounds`, `.remaining` and
`.best`. Score the initial population through `run`, then implement
generations yourself. Use `random.Random(seed)` for search randomness.

For each target choose three distinct OTHER members from a snapshot of
the preceding generation. Form `r1 + F*(r2-r3)`, clip to bounds, and keep
the donor if no worse than the target. At T1/T2 use the whole donor.
Extend this SAME function with coordinate crossover at T3.
Stop exactly at the budget, even if it ends within a generation.


In [ ]:
def student_de(run, population, *, F=0.8, CR=1.0, seed=0):
    # Implement your DE here, using only standard Python and supplied scoring.
    raise NotImplementedError("Write your DE loop before enabling experiments.")

RUN_T1 = False
t1_runs = {}
if RUN_T1:
    for objective in ("MAE", "MedAE"):
        run = run_fitting(student_de, line_data, objective, budget=FIT_BUDGET)
        t1_runs[objective] = run
        run.save(OUTPUT / f"T1_{objective}")
        print(objective, fit_metrics(line_data, run.best))
        plot_progress([run], f"T1: {objective}")
        plt.show()
    plot_fit(line_data, {name: run.best for name, run in t1_runs.items()})
    plt.show()
    plot_landscapes(line_data, {name: run.best for name, run in t1_runs.items()})
    plt.show()
else:
    print("T1 experiments wait for your student_de implementation.")


## T2. More parameters, noise and outliers

Reuse DE with CR=1. Fit the cubic and quintic, each with 0% and 20%
one-sided outliers, separately for MAE and MedAE: eight fits. Fixed
population 40, F=0.8, seed 0, your chosen `FIT_BUDGET` each. Polynomial
coefficients are lowest power first. The independent random streams
keep observation locations and noise identical when contamination changes.

Plot the four datasets first. The optional exploratory generator changes
curve coefficients, `noise`, `sampling` (uniform/gap/clustered),
`noise_pattern` (constant/increasing), `outlier_fraction` or
`outlier_direction` (up/mixed). Keep the required datasets unchanged.


In [ ]:
def t2_datasets():
    return [polynomial_data(case, outlier_fraction=fraction)
            for case in ("cubic", "quintic") for fraction in (0.0, 0.2)]


def exploratory_polynomial(coefficients, **options):
    coefficients = list(coefficients)
    if not 2 <= len(coefficients) <= 6 or not np.all(np.isfinite(coefficients)):
        raise ValueError("Supply 2-6 finite coefficients.")
    limit = max(6, 1 + max(abs(value) for value in coefficients))
    return observations("polynomial", coefficients, [(-limit, limit)] * len(coefficients),
                        name=f"Exploratory degree {len(coefficients) - 1}", **options)


def plot_t2_gallery(datasets, fits=None):
    datasets = list(datasets)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
    if len(datasets) != 4:
        raise ValueError("Supply the four T2 datasets.")
    for index, (data, ax) in enumerate(zip(datasets, axes.flat)):
        plot_fit(data, (fits or {}).get(index, {}), ax=ax)
    fig.suptitle("T2: same observations, changed degree or contamination")
    return fig


In [ ]:
t2_data = t2_datasets()
gallery = plot_t2_gallery(t2_data)
gallery.savefig(OUTPUT / "T2_datasets.png", dpi=140)
plt.show()

# Optional bolder example, NOT another required fit.
exploratory = exploratory_polynomial([0.1, -2.4, 0.2, 2.7], n=40,
    noise=0.25, sampling="gap", noise_pattern="increasing",
    outlier_fraction=0.1, outlier_direction="mixed")
plot_fit(exploratory)
plt.savefig(OUTPUT / "T2_exploration.png", dpi=140)
plt.show()


In [ ]:
RUN_T2 = False
t2_runs, t2_fits = {}, {}
if RUN_T2:
    for index, data in enumerate(t2_data):
        t2_fits[index] = {}
        for objective in ("MAE", "MedAE"):
            run = run_fitting(student_de, data, objective, budget=FIT_BUDGET)
            t2_runs[index, objective] = run
            t2_fits[index][objective] = run.best
            run.save(OUTPUT / f"T2_{index}_{objective}")
            print(data["name"], data["outlier_fraction"], objective, fit_metrics(data, run.best))
    plot_t2_gallery(t2_data, t2_fits)
    plt.show()
    for objective in ("MAE", "MedAE"):
        plot_progress([t2_runs[3, objective]], f"T2: contaminated quintic, {objective}")
        plt.show()
else:
    print("T2 experiments reuse your DE when RUN_T2 is True.")


## T3. Introduce crossover, one parameter at a time

Extend `student_de`: copy each coordinate from the donor with probability
CR, otherwise from the target. Force ONE random coordinate to come from
the donor. CR=1 uses the whole donor; CR=0 copies exactly one coordinate.
Donors, bounds and selection remain unchanged.

Start with the contaminated quintic and MedAE. As the course page allows,
if that dataset does not illustrate the point well, make or choose another
one and set `t3_data` below. Keep that chosen dataset fixed across all
settings and seeds; describe it in the report. A cached T2 run is reused
only when the actual training data, bounds, model and settings match.

Four CR values x three seeds
give 12 runs. Then vary EITHER population or F at CR=0.7: two additional
settings x three seeds give six new runs. The settings table below
constructs all 18 configurations without duplicating the baseline.
Initialisation uses a separate RNG, giving identical initial populations
within each seed when population size is unchanged.

Keep your chosen `FIT_BUDGET` fixed across all settings and seeds.
Target MedAE is 0.10, a milestone rather than a grading threshold.
Keep failures. Report median/range and all trials;
three seeds do not establish strong statistical significance.


In [ ]:
def t3_configurations(extra="population", budget=8000):
    """12 crossover trials + 6 extra trials, reusing the baseline."""
    if extra not in ("population", "F"):
        raise ValueError("Choose population or F, not both.")
    minimum = 80 if extra == "population" else 40
    if not isinstance(budget, int) or budget < minimum:
        raise ValueError(f"Budget must be an integer >= {minimum} to cover every initial population.")
    settings = [{"population": 40, "F": 0.8, "CR": cr} for cr in (1.0, 0.7, 0.3, 0.0)]
    for value in ((20, 80) if extra == "population" else (0.4, 1.2)):
        item = {"population": 40, "F": 0.8, "CR": 0.7}
        item[extra] = value
        settings.append(item)
    return [dict(item, seed=seed, budget=budget) for item in settings for seed in (0, 1, 2)]


def grouped_summary(runs, target=0.10):
    """Median/range of final fitness; retain each failed target attempt."""
    from collections import defaultdict
    from statistics import median

    groups = defaultdict(list)
    for run in runs:
        key = tuple(run.settings[key] for key in ("population", "F", "CR"))
        groups[key].append(run)
    result = []
    for (population, F, CR), trials in groups.items():
        scores = [run.best_fitness for run in trials]
        row = {"population": population, "F": F, "CR": CR, "runs": len(trials),
               "median": median(scores), "min": min(scores), "max": max(scores),
               "target_hits": sum(run.summary(target)["reached"] for run in trials)}
        result.append(row)
        print(row)
    return result


In [ ]:
t3_data = t2_data[3]  # May replace with another dataset, as allowed on the course page
t3_digest = fitting_data_digest(t3_data)
configurations = t3_configurations(extra="population", budget=FIT_BUDGET)  # or "F", not both
print("Number of planned runs:", len(configurations))
for config in configurations:
    print(config)

RUN_T3 = False
t3_runs = []
if RUN_T3:
    for index, config in enumerate(configurations):
        label = f"NP={config['population']}, F={config['F']}, CR={config['CR']}, s={config['seed']}"
        # Reuse the identical T2 whole-donor run if it is available.
        cached = t2_runs.get((3, "MedAE"))
        if (config == dict(population=40, F=0.8, CR=1.0, seed=0, budget=FIT_BUDGET)
                and cached is not None and cached.budget == FIT_BUDGET
                and cached.seed == 0 and cached.objective_name == "MedAE"
                and cached.settings.get("training_sha256") == t3_digest
                and all(cached.settings.get(key) == config[key]
                        for key in ("population", "F", "CR"))):
            run = cached
        else:
            run = run_fitting(student_de, t3_data, "MedAE", label=label, **config)
        t3_runs.append(run)
        run.save(OUTPUT / f"T3_{index}", target=0.10)
    trial_rows = experiment_table(t3_runs, target=0.10)
    grouped_rows = grouped_summary(t3_runs)
    plot_progress([run for run in t3_runs if run.seed == 0], "T3: seed 0; tables include every seed")
    plt.show()


## T4. Unknown frequencies: a harder landscape

The one-wave model has six parameters:
`[c0,c1,c2,A1,w1,phi1]`. The two-wave model has nine, adding
`[A2,w2,phi2]`. They describe a quadratic trend plus one or two sine terms.
Frequencies are radians per x-unit. Bounds: trend coefficients [-2,2],
amplitudes [0.2,2], phases [-pi,pi]. Frequencies are in [pi,8*pi] for
one wave and [pi,16*pi] for the harder two-wave case.

Use MAE, population 40, F=0.8, CR=0.7, your chosen `HARD_BUDGET` and seeds 0/1/2
on EACH case. Plot unsuccessful runs too. Unknown frequency creates
competing fits; higher polynomial degree by itself is not evidence of
multiple least-squares minima. Recover curves, not unique parameter lists:
phase wrapping and exchanging sine terms can describe the same curve.


In [ ]:
def wave_data(waves=1, **options):
    if waves not in (1, 2):
        raise ValueError("Use one or two sine terms.")
    truth = [0.3, -0.5, 0.3, 1.2, 3.2 * np.pi, 0.4]
    if waves == 2:
        truth[4] = 6.1 * np.pi
        truth += [0.8, 11.7 * np.pi, -0.9]
    max_frequency = (8 if waves == 1 else 16) * np.pi
    bounds = [(-2, 2)] * 3 + [(0.2, 2.0), (np.pi, max_frequency), (-np.pi, np.pi)] * waves
    model = "one-wave" if waves == 1 else "two-wave"
    options.setdefault("n", 180)
    return observations(model, truth, bounds, name=model, **options)


In [ ]:
t4_data = [wave_data(1), wave_data(2)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), layout="constrained")
for data, ax in zip(t4_data, axes):
    plot_fit(data, ax=ax)
    print(data["name"], "parameter count:", len(data["bounds"]), "bounds:", data["bounds"])
fig.savefig(OUTPUT / "T4_wave_data.png", dpi=140)
plt.show()

RUN_T4 = False
t4_runs = []
if RUN_T4:
    for data in t4_data:
        fits, case_runs = {}, []
        for seed in (0, 1, 2):
            run = run_fitting(student_de, data, "MAE", seed=seed, CR=0.7, budget=HARD_BUDGET)
            t4_runs.append(run)
            case_runs.append(run)
            fits[f"Seed {seed}"] = run.best
            run.save(OUTPUT / f"T4_{data['name']}_{seed}")
            print(data["name"], seed, fit_metrics(data, run.best))
        plot_fit(data, fits)
        plt.show()
        plot_progress(case_runs, f"T4: {data['name']}")
        plt.show()


## Report and bonuses

Select the figures and tables needed to explain your results; do not dump
all raw output into the report. Logs and settings belong with your code.
B1 changes your own optimiser under the same total budget. B2 compares
packages on the SAME problem and objective; only B2 permits packaged
optimisation. Use a fresh `FitnessRun` per method/seed, count each candidate
in a vectorised batch, and disable or explicitly budget extra polishing.
Do not silently continue when `BudgetExhausted` is raised.
